Create a Bi-partite weighted temporal graph:
- actor (name) - movie (name)

label on actor : gander, age 
label on movie : genre

In [1]:
import pandas as pd 


In [2]:
df = pd.read_csv("data/processed/char_movie_actor_86_07.csv")

df.columns.to_list()

['Unnamed: 0.1',
 'Unnamed: 0',
 'wiki_id',
 'freebase_movie_id',
 'release_date',
 'char_name',
 'actor_dob',
 'actor_gender',
 'actor_height',
 'actor_ethnicity',
 'actor_name',
 'actor_age',
 'freebase_char_id',
 'freebase_actor_id',
 'freebase_id_2',
 'movie_name',
 'year']

create graph with igraph (snapshots)
igraph does not allow bipartite graphs, so we will use a flag 0 or 1 for actor or films

In [ ]:
# !pip install igraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 4.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [igraph]


In [8]:
import os
import json
import csv
import pickle
import igraph as ig
import pandas as pd

# 1. Creiamo un dizionario veloce per mappare: wiki_id -> lista di generi
# Leggiamo direttamente il file dei metadati dei film
movie_genres_dict = {}

print("Lettura e parsing dei generi da movie.metadata.tsv...")
with open("data/MovieSummaries/movie.metadata.tsv", 'r', encoding='utf-8') as f:
    reader = csv.reader(f, delimiter='\t')
    for row in reader:
        if len(row) > 8:
            wiki_id = int(row[0])
            genre_json = row[8] # L'ultima colonna contiene il dizionario dei generi
            try:
                # Trasformiamo la stringa JSON in un vero dizionario Python
                genres_data = json.loads(genre_json)
                # Estraiamo solo i valori (i nomi dei generi)
                genres_list = list(genres_data.values())
                movie_genres_dict[wiki_id] = genres_list
            except json.JSONDecodeError:
                movie_genres_dict[wiki_id] = []

print(f"Mappati i generi per {len(movie_genres_dict)} film.")

# now the graph
df = pd.read_csv("data/processed/char_movie_actor_86_07.csv")

output_dir = "data/processed/temporal_graphs"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

grouped = df.groupby('year')

print("\nInizio generazione dei grafi annuali con iGraph e Generi...")

for year, group in grouped:
    unique_actors = group['actor_name'].dropna().unique().tolist()
    unique_movies = group['movie_name'].dropna().unique().tolist()
    
    all_nodes = unique_actors + unique_movies
    node_to_id = {name: i for i, name in enumerate(all_nodes)}
    
    # Mappatura veloce wiki_id -> movie_name per l'anno corrente
    movie_name_to_wiki = group.set_index('movie_name')['wiki_id'].to_dict()
    
    g = ig.Graph()
    g.add_vertices(len(all_nodes))
    
    g.vs["name"] = all_nodes
    g.vs["type"] = [False if i < len(unique_actors) else True for i in range(len(all_nodes))]
    
    # Mappe per gli attributi degli attori
    actor_gender_map = group.set_index('actor_name')['actor_gender'].to_dict()
    actor_age_map = group.set_index('actor_name')['actor_age'].to_dict()
    
    # Popoliamo gli attributi dei nodi
    g.vs["gender"] = [actor_gender_map.get(name, None) if not is_movie else None for name, is_movie in zip(all_nodes, g.vs["type"])]
    g.vs["age"] = [int(actor_age_map.get(name, 0)) if not is_movie else None for name, is_movie in zip(all_nodes, g.vs["type"])]
    
    # Recuperiamo la lista dei generi usando il wiki_id associato al nome del film
    movie_genres_attr = []
    for name, is_movie in zip(all_nodes, g.vs["type"]):
        if is_movie:
            w_id = movie_name_to_wiki.get(name)
            # Salviamo i generi uniti da virgola (es: "Action, Sci-Fi") o come lista. 
            # La stringa unita da virgola è più comoda per alcuni export, la lista è più pulita. Scegliamo la lista:
            movie_genres_attr.append(movie_genres_dict.get(w_id, []))
        else:
            movie_genres_attr.append(None)
            
    g.vs["genre"] = movie_genres_attr
    
    # Costruzione archi pesati
    edges_df = group.groupby(['actor_name', 'movie_name']).size().reset_index(name='weight')
    edges = []
    weights = []
    for _, row in edges_df.iterrows():
        edges.append((node_to_id[row['actor_name']], node_to_id[row['movie_name']]))
        weights.append(row['weight'])
        
    g.add_edges(edges)
    g.es["weight"] = weights
    g.es["year"] = int(year)
    
    # Salvataggio
    file_path = os.path.join(output_dir, f"graph_{int(year)}.pkl")
    with open(file_path, 'wb') as f:
        pickle.dump(g, f)

print("\n[FINISH] Grafi temporali rigenerati con successo includendo i multi-generi dei film!")

Lettura e parsing dei generi da movie.metadata.tsv...
Mappati i generi per 81741 film.

Inizio generazione dei grafi annuali con iGraph e Generi...

[FINISH] Grafi temporali rigenerati con successo includendo i multi-generi dei film!


In [ ]:
import os
import pickle
import igraph as ig

# 1. Definiamo il percorso del file di test
test_year = 2002
file_path = f"data/processed/temporal_graphs/graph_{test_year}.pkl"

if not os.path.exists(file_path):
    print(f"Errore: Il file per l'anno {test_year} non esiste. Controlla il percorso!")
else:
    # 2. Carichiamo il grafo dal file Pickle
    with open(file_path, 'rb') as f:
        g_test = pickle.load(f)
    
    print(f"=== TEST VERIFICA GRAFO ANNUALE ({test_year}) ===")
    print(f"Totale Nodi nel grafo: {g_test.vcount()}")
    print(f"Totale Archi nel grafo: {g_test.ecount()}\n")
    
    # 3. VERIFICA NODI (Controlliamo i primi 3 Attori e i primi 3 Film)
    print("--- Verifica Attributi dei Nodi ---")
    
    # Modo ottimizzato iGraph per selezionare i nodi in base agli attributi
    attori_indices = g_test.vs.select(type=False)[:3]
    film_indices = g_test.vs.select(type=True)[:3]
    
    print("Campione Nodi ATTORI (type=False):")
    for v in attori_indices:
        print(f"  - Nome: {v['name']} | Genere: {v['gender']} | Età: {v['age']} | Type: {v['type']}")
        
    print("\nCampione Nodi FILM (type=True):")
    for v in film_indices:

        year_val = g_test.es[0]['year'] if g_test.ecount() > 0 else test_year
        print(f"  - Titolo: {v['name']} | Genere Film: {v['genre']} | Anno: {year_val} | Type: {v['type']}")
    
    # 4. VERIFICA ARCHI (Controlliamo i primi 3 collegamenti)
    print("\n--- Verifica Attributi degli Archi ---")
    if g_test.ecount() > 0:
        for i in range(min(3, g_test.ecount())):
            edge = g_test.es[i]
            source_node = g_test.vs[edge.source]['name']
            target_node = g_test.vs[edge.target]['name']
            print(f"  - Arco {i}: {source_node} <---> {target_node}")
            print(f"    [Attributi] Peso (Ruoli): {edge['weight']} | Anno dell'arco: {edge['year']}")
    else:
        print("  Attenzione: Non ci sono archi in questo grafo!")

    # 5. CONTROLLO DI INTEGRITÀ BIPARTITA
    bipartite_errors = 0
    for edge in g_test.es:
        if g_test.vs[edge.source]['type'] == g_test.vs[edge.target]['type']:
            bipartite_errors += 1
            
    print(f"\n--- Controllo Integrità Bipartita ---")
    if bipartite_errors == 0:
        print("✅ Successo! Il grafo è strutturato perfettamente: non ci sono archi tra attore-attore o film-film.")
    else:
        print(f"❌ Errore: Trovati {bipartite_errors} archi non bipartiti.")


=== TEST VERIFICA GRAFO ANNUALE (2002) ===
Totale Nodi nel grafo: 3258
Totale Archi nel grafo: 3454

--- Verifica Attributi dei Nodi ---
Campione Nodi ATTORI (type=False):
  - Nome: Martina Stella | Genere: F | Età: 17 | Type: False
  - Nome: Cesare Cremonini | Genere: M | Età: 21 | Type: False
  - Nome: Chiara Sani | Genere: F | Età: 38 | Type: False

Campione Nodi FILM (type=True):
  - Titolo: Un amore perfetto | Genere Film: ['Romantic comedy'] | Anno: 2002 | Type: True
  - Titolo: Resident Evil | Genere Film: ['Thriller', 'Science Fiction', 'Horror', 'Adventure', 'Doomsday film', 'Sci-Fi Horror', 'Creature Film', 'Plague', 'Zombie Film', 'Action'] | Anno: 2002 | Type: True
  - Titolo: It's a Very Merry Muppet Christmas Movie | Genere Film: ["Children's/Family", 'Holiday Film', 'Comedy', 'Heavenly Comedy', "Children's Fantasy"] | Anno: 2002 | Type: True

--- Verifica Attributi degli Archi ---
  - Arco 0: A.K Hangal <---> Shararat
    [Attributi] Peso (Ruoli): 1 | Anno dell'arco: 200

In [15]:
# further test

import os
import pickle
import igraph as ig

# 1. Definiamo il percorso del file di test
test_year = 2002
file_path = f"data/processed/temporal_graphs/graph_{test_year}.pkl"

if not os.path.exists(file_path):
    print(f"Errore: Il file per l'anno {test_year} non esiste. Controlla il percorso!")
else:
    # 2. Carichiamo il grafo dal file Pickle
    with open(file_path, 'rb') as f:
        g_test = pickle.load(f)


# attori che hanno peso diverso da 1
# 6. TEST FUNZIONALE: Trova archi con peso diverso da 1
    print("\n--- Verifica Archi con Peso > 1 (Attori con più ruoli nello stesso film) ---")
    
    # Selezioniamo gli archi che hanno un peso maggiore di 1
    archi_pesanti = g_test.es.select(weight_gt=1)
    
    if len(archi_pesanti) > 0:
        print(f"Trovati {len(archi_pesanti)} collegamenti con peso maggiore di 1:")
        # Ne mostriamo al massimo 10 per non intasare l'output
        for i, edge in enumerate(archi_pesanti[:10]):
            actor_name = g_test.vs[edge.source]['name']
            movie_name = g_test.vs[edge.target]['name']
            print(f"  - {actor_name} ha {edge['weight']} ruoli in '{movie_name}'")
        if len(archi_pesanti) > 10:
            print(f"  ... e altri {len(archi_pesanti) - 10} archi pesanti.")
    else:
        print("  Nessun attore ha più di un ruolo nello stesso film per questo anno (tutti i pesi sono uguali a 1).")


    # 6. TEST FUNZIONALE: Trova attori presenti in più film (Degree > 1)
    print(f"\n--- Attori presenti in PIÙ FILM nel {test_year} ---")
    
    # Selezioniamo solo i nodi attore (type=False)
    attori = g_test.vs.select(type=False)
    
    # Troviamo quelli che hanno un degree (numero di collegamenti a film diversi) > 1
    attori_multi_film = [v for v in attori if v.degree() > 1]
    
    if len(attori_multi_film) > 0:
        print(f"Trovati {len(attori_multi_film)} attori che lavorano in più film contemporaneamente:")
        
        # Ordiniamo per numero di film decrescente e mostriamo i primi 10
        attori_ordinati = sorted(attori_multi_film, key=lambda x: x.degree(), reverse=True)
        
        for v in attori_ordinati[:10]:
            # Recuperiamo i titoli dei film a cui è collegato
            film_collegati = [g_test.vs[neighbor]['name'] for neighbor in g_test.neighbors(v.index)]
            print(f"  - {v['name']}: lavora in {v.degree()} film {film_collegati}")
    else:
        print(f"  Nessun attore è presente in più di un film nel {test_year}.")



--- Verifica Archi con Peso > 1 (Attori con più ruoli nello stesso film) ---
Trovati 41 collegamenti con peso maggiore di 1:
  - Adam Godley ha 2 ruoli in 'War Game'
  - Ahn Sung-ki ha 2 ruoli in 'The Romantic President'
  - Alan Tudyk ha 2 ruoli in 'Ice Age'
  - Alex 'Alien' Russell ha 2 ruoli in 'Ken Russell's Fall of the Louse of Usher'
  - Andy Richter ha 2 ruoli in 'Big Trouble'
  - Blair Underwood ha 2 ruoli in 'Full Frontal'
  - Brent Spiner ha 2 ruoli in 'Star Trek Nemesis'
  - Chad Lowe ha 2 ruoli in 'Unfaithful'
  - Chris Rock ha 3 ruoli in 'Bad Company'
  - Chris Wedge ha 2 ruoli in 'Ice Age'
  ... e altri 31 archi pesanti.

--- Attori presenti in PIÙ FILM nel 2002 ---
Trovati 541 attori che lavorano in più film contemporaneamente:
  - Johnny Lever: lavora in 7 film ['Wayward, Crazy, Insane', 'Yeh Mohabbat Hai', 'Annarth', 'Jaani Dushman: Ek Anokhi Kahani', "Come, Let's Fall In Love", 'Humraaz', 'Pyaar Diwana Hota Hai']
  - Rajpal Yadav: lavora in 7 film ['Tumko Na Bhool Pa

In [16]:
# now the actor projection

import os
import pickle
import igraph as ig

# Percorsi delle cartelle
input_dir = "data/processed/temporal_graphs"
output_dir = "data/processed/actor_projections"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print("Inizio proiezione dei grafi degli attori anno per anno...")

# Scorriamo i file pkl dei grafi bipartiti originari
for filename in sorted(os.listdir(input_dir)):
    if filename.endswith(".pkl"):
        year = filename.split("_")[1].split(".")[0]
        file_path = os.path.join(input_dir, filename)
        
        with open(file_path, 'rb') as f:
            g_bipartite = pickle.load(f)
            
        # .bipartite_projection() restituisce una tupla di due grafi:
        # il primo (index 0) è la proiezione per i nodi con type=False (gli ATTORI)
        # il secondo (index 1) è la proiezione per i nodi con type=True (i FILM)
        g_actors, _ = g_bipartite.bipartite_projection(multiplicity=True)
        
        # 'multiplicity=True' fa una cosa fantastica: assegna all'attributo 'weight' dell'arco
        # il numero di film che i due attori hanno condiviso in quell'anno.
        
        # Aggiungiamo un attributo temporale globale al grafo per sicurezza
        g_actors["year"] = int(year)
        
        # Rimuoviamo l'attributo 'type' che ora è superfluo (sono tutti attori)
        if "type" in g_actors.vs.attributes():
            del g_actors.vs["type"]
            
        # Salviamo la proiezione degli attori in un nuovo file Pickle
        output_path = os.path.join(output_dir, f"actors_projected_{year}.pkl")
        with open(output_path, 'wb') as f:
            pickle.dump(g_actors, f)
            
        print(f"-> Anno {year}: Proiettati {g_actors.vcount()} attori con {g_actors.ecount()} co-occorrenze.")

print("\n[FINISH] Tutte le proiezioni annuali degli attori sono state salvate!")

Inizio proiezione dei grafi degli attori anno per anno...
-> Anno 1986: Proiettati 866 attori con 3615 co-occorrenze.
-> Anno 1987: Proiettati 1001 attori con 4139 co-occorrenze.
-> Anno 1988: Proiettati 897 attori con 3720 co-occorrenze.
-> Anno 1989: Proiettati 958 attori con 4122 co-occorrenze.
-> Anno 1990: Proiettati 950 attori con 4523 co-occorrenze.
-> Anno 1991: Proiettati 943 attori con 3991 co-occorrenze.
-> Anno 1992: Proiettati 1061 attori con 4352 co-occorrenze.
-> Anno 1993: Proiettati 1138 attori con 5238 co-occorrenze.
-> Anno 1994: Proiettati 1328 attori con 6978 co-occorrenze.
-> Anno 1995: Proiettati 1294 attori con 6548 co-occorrenze.
-> Anno 1996: Proiettati 1726 attori con 10364 co-occorrenze.
-> Anno 1997: Proiettati 1963 attori con 11980 co-occorrenze.
-> Anno 1998: Proiettati 2261 attori con 13859 co-occorrenze.
-> Anno 1999: Proiettati 2262 attori con 14418 co-occorrenze.
-> Anno 2000: Proiettati 2372 attori con 13914 co-occorrenze.
-> Anno 2001: Proiettati 26